In [ ]:
%pip install yfinance azure-eventhub azure-identity

In [8]:
from datetime import datetime, timedelta

import pandas as pd
import yfinance as yf
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import TimestampType

StatementMeta(, 30493884-9443-4be7-bd3b-97a0d2a9c37d, 20, Finished, Available, Finished, False)

In [9]:
TABLE_NAME = "DailyStocks"

STOCKS = [
    'MSFT', 'AAPL', 'VZ',   'SOFI', 'AGNC', 'ELPW', 'U',
    'GME',  'NBIS', 'AMAT', 'APP',  'RBLX', 'SHOP', 'LRCX',
    'ROKU', 'DKNG', 'CL',   'CHWY',
]

TODAY          = datetime.today().strftime('%Y-%m-%d')
INGESTION_DATE = datetime.today().strftime('%Y-%m-%d')
SIXTY_DAYS_AGO = (datetime.today() - timedelta(days=60)).strftime('%Y-%m-%d')

# Used for filtering the daily history — tz-aware to match yfinance index
SIXTY_DAYS_AGO_TS = pd.Timestamp(datetime.today() - timedelta(days=60)).tz_localize('America/New_York')

spark = SparkSession.builder.getOrCreate()

StatementMeta(, 30493884-9443-4be7-bd3b-97a0d2a9c37d, 21, Finished, Available, Finished, False)

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — Full daily history (max period, up to 60 days ago)
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("PHASE 1: Downloading full daily history...")
print("=" * 60)

all_data = {}

for stock in STOCKS:
    ticker  = yf.Ticker(stock)
    history = ticker.history(period='max')
    all_data[stock] = history[history.index < SIXTY_DAYS_AGO_TS]
    print(f"  {stock}: {len(all_data[stock]):,} daily rows")

# Combine
df_daily = pd.concat(all_data, axis=0)
df_daily.index.names = ['StockSymbol', 'TradeDate']
df_daily = df_daily.reset_index()

# Clean columns
df_daily.columns = (df_daily.columns
                    .str.replace('#', '', regex=False)
                    .str.replace(' ', '', regex=False))

df_daily['TradeDate']     = pd.to_datetime(df_daily['TradeDate']).dt.tz_localize(None)
df_daily['IngestionDate'] = INGESTION_DATE
df_daily['Granularity']   = '1d'
df_daily['Volume'] = df_daily['Volume'].astype('float64')

df_daily = df_daily[[
    'StockSymbol', 'TradeDate',
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Dividends', 'StockSplits', 'IngestionDate', 'Granularity',
]]

print(f"\nPhase 1 complete: {len(df_daily):,} daily rows across {df_daily['StockSymbol'].nunique()} stocks")


StatementMeta(, 30493884-9443-4be7-bd3b-97a0d2a9c37d, 22, Finished, Available, Finished, False)

PHASE 1: Downloading full daily history...
  MSFT: 10,044 daily rows
  AAPL: 11,370 daily rows
  VZ: 10,626 daily rows
  SOFI: 1,270 daily rows
  AGNC: 4,451 daily rows
  ELPW: 686 daily rows
  U: 1,343 daily rows
  GME: 6,025 daily rows
  NBIS: 315 daily rows
  AMAT: 11,558 daily rows
  APP: 1,200 daily rows
  RBLX: 1,225 daily rows
  SHOP: 2,686 daily rows
  LRCX: 10,512 daily rows
  ROKU: 2,091 daily rows
  DKNG: 1,634 daily rows
  CL: 13,295 daily rows
  CHWY: 1,662 daily rows

Phase 1 complete: 91,993 daily rows across 18 stocks


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2 — 5m intraday for the last 60 days
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("PHASE 2: Downloading 5m intraday data (last 60 days)...")
print("=" * 60)

raw_5m = yf.download(STOCKS, start=SIXTY_DAYS_AGO, end=TODAY, interval="5m")

if raw_5m.empty:
    raise ValueError("yfinance returned empty 5m data. Check date range or ticker symbols.")

# Flatten multi-level columns → "Metric_Symbol"
raw_5m.columns = ['_'.join(col).strip() for col in raw_5m.columns.values]

# Wide → long
df_long = raw_5m.reset_index().rename(columns={raw_5m.reset_index().columns[0]: 'Datetime'})
df_long['Datetime'] = df_long['Datetime'].dt.tz_convert('UTC').dt.tz_localize(None)

df_long = pd.melt(
    df_long,
    id_vars=['Datetime'],
    value_vars=[col for col in raw_5m.columns],
    var_name='Temp',
    value_name='Value',
)

# Split "Close_AAPL" → Metric="Close", StockSymbol="AAPL"
# n=1 guards against metric names containing underscores
df_long[['Metric', 'StockSymbol']] = df_long['Temp'].str.split('_', n=1, expand=True)
df_long = df_long.drop(columns=['Temp'])
df_long = df_long.rename(columns={'Datetime': 'TradeDate'})

# Long → wide (one row per TradeDate × StockSymbol)
df_5m = (
    df_long
    .pivot_table(index=['TradeDate', 'StockSymbol'], columns='Metric', values='Value')
    .reset_index()
    .rename_axis(None, axis=1)
)

df_5m['IngestionDate'] = INGESTION_DATE
df_5m['Dividends']     = 0
df_5m['StockSplits']   = 0
df_5m['Granularity']   = '5m'
df_5m['Volume'] = df_5m['Volume'].astype('float64')

df_5m = df_5m[[
    'StockSymbol', 'TradeDate',
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Dividends', 'StockSplits', 'IngestionDate', 'Granularity',
]]

print(f"\nPhase 2 complete: {len(df_5m):,} 5m rows across {df_5m['StockSymbol'].nunique()} stocks")


StatementMeta(, 30493884-9443-4be7-bd3b-97a0d2a9c37d, 23, Finished, Available, Finished, False)


PHASE 2: Downloading 5m intraday data (last 60 days)...

Phase 2 complete: 56,634 5m rows across 18 stocks


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# COMBINE & WRITE
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("Combining and writing to Delta...")
print("=" * 60)

FLOAT_COLS = ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'StockSplits']

df_daily[FLOAT_COLS] = df_daily[FLOAT_COLS].astype('float64')
df_5m[FLOAT_COLS]    = df_5m[FLOAT_COLS].astype('float64')

df_all = pd.concat([df_daily, df_5m], axis=0, ignore_index=True)

print(f"Total rows to write: {len(df_all):,}")
print(f"  Daily (1d) rows : {len(df_daily):,}")
print(f"  Intraday (5m) rows: {len(df_5m):,}")

(
    spark.createDataFrame(df_all)
         .withColumn('TradeDate', F.col('TradeDate').cast(TimestampType()))
         .write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .partitionBy("StockSymbol")
         .saveAsTable(TABLE_NAME)
)

final_count = spark.table(TABLE_NAME).count()
print(f"\n✅ Initial load complete. '{TABLE_NAME}' contains {final_count:,} rows.")


StatementMeta(, 30493884-9443-4be7-bd3b-97a0d2a9c37d, 25, Finished, Available, Finished, False)


Combining and writing to Delta...
Total rows to write: 148,627
  Daily (1d) rows : 91,993
  Intraday (5m) rows: 56,634

✅ Initial load complete. 'DailyStocks' contains 148,627 rows.
